# M6P2 Interactions and Study Design for DESeq2

For Module 4, in our first practice we ran a DESeq2 analysis on the 12 samples from the [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) data set.
This was done by performing the analysis across the root (OPT3 vs Col-0 and Cd-treated vs untreated replete) samples. The table below shows the samples within the dataset, and their conditions: 

| Sample ID   | Replicate Name | Genotype | Tissue | Biological Rep | Technical Rep |
|------------|----------------|----------|--------|----------------|---------------|
| GSM2090459 | CRN1-1         | Col-0    | Root   | 1              | 1             |
| GSM2090460 | CRN1-2         | Col-0    | Root   | 1              | 2             |
| GSM2090461 | CRN1-3         | Col-0    | Root   | 1              | 3             |
| GSM2090462 | CRN2-1         | Col-0    | Root   | 2              | 1             |
| GSM2090463 | CRN2-2         | Col-0    | Root   | 2              | 2             |
| GSM2090464 | CRN2-3         | Col-0    | Root   | 2              | 3             |
| GSM2090465 | CRN3-1         | Col-0    | Root   | 3              | 1             |
| GSM2090466 | CRN3-2         | Col-0    | Root   | 3              | 2             |
| GSM2090467 | CRN3-3         | Col-0    | Root   | 3              | 3             |
| GSM2090468 | CLN1-1         | Col-0    | Leaf   | 1              | 1             |
| GSM2090469 | CLN1-2         | Col-0    | Leaf   | 1              | 2             |
| GSM2090470 | CLN1-3         | Col-0    | Leaf   | 1              | 3             |
| GSM2090471 | CLN2-1         | Col-0    | Leaf   | 2              | 1             |
| GSM2090472 | CLN2-2         | Col-0    | Leaf   | 2              | 2             |
| GSM2090473 | CLN2-3         | Col-0    | Leaf   | 2              | 3             |
| GSM2090474 | CLN3-1         | Col-0    | Leaf   | 3              | 1             |
| GSM2090475 | CLN3-2         | Col-0    | Leaf   | 3              | 2             |
| GSM2090476 | CLN3-3         | Col-0    | Leaf   | 3              | 3             |
| GSM2090477 | ORN1-1         | opt3-2   | Root   | 1              | 1             |
| GSM2090478 | ORN1-2         | opt3-2   | Root   | 1              | 2             |
| GSM2090479 | ORN1-3         | opt3-2   | Root   | 1              | 3             |
| GSM2090480 | ORN2-1         | opt3-2   | Root   | 2              | 1             |
| GSM2090481 | ORN2-2         | opt3-2   | Root   | 2              | 2             |
| GSM2090482 | ORN2-3         | opt3-2   | Root   | 2              | 3             |
| GSM2090483 | ORN3-1         | opt3-2   | Root   | 3              | 1             |
| GSM2090484 | ORN3-2         | opt3-2   | Root   | 3              | 2             |
| GSM2090485 | ORN3-3         | opt3-2   | Root   | 3              | 3             |
| GSM2090486 | OLN1-1         | opt3-2   | Leaf   | 1              | 1             |
| GSM2090487 | OLN1-2         | opt3-2   | Leaf   | 1              | 2             |
| GSM2090488 | OLN1-3         | opt3-2   | Leaf   | 1              | 3             |
| GSM2090489 | OLN2-1         | opt3-2   | Leaf   | 2              | 1             |
| GSM2090490 | OLN2-2         | opt3-2   | Leaf   | 2              | 2             |
| GSM2090491 | OLN2-3         | opt3-2   | Leaf   | 2              | 3             |
| GSM2090492 | OLN3-1         | opt3-2   | Leaf   | 3              | 1             |
| GSM2090493 | OLN3-2         | opt3-2   | Leaf   | 3              | 2             |
| GSM2090494 | OLN3-3         | opt3-2   | Leaf   | 3              | 3             |

**According to the [Metadata](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE79nnn/GSE79275/miniml/), this is how the data was processed:**
- Read Cleaning: Reads were cleaned in R using ShortRead::TrimEnds, removing bases from each end until the remaining read has bases with p-values all over 0.1.  Afterwards, reads shorter than 25bp were removed.
- Read Alignment: Reads were aligned to the TAIR10 genome using Tophat2 using default parameters except -g 1, specifying to only retain reads with one valid alignment
- Read counting: Reads were counted in R using GenomicAlignments::summarizeOverlaps with the TAIR10 gff as input and the counting mode of 'union'
- FPKM calculations: pseudo-counts (generated from edgeR's library size normalization) were divided by the gene length (in kb) found in the TAIR 10 gff.

Today, we will specifically focus on the interaction considerations for a DESeq 2 analysis, as well as provide a review of how these analyses are performed. 

**We will cover:**
- Review of importing count data to R for DESeq2 usage
- How DESeq2 handles interactions
- The effects of different interactions within our study design for DESeq2

```{admonition} Adapted from
:class: tip
**Source:** [Introduction to DGE](https://hbctraining.github.io/Intro-to-DGE/), Harvard Chan Bioinformatics Core (HBC) training team, [hbctraining/Intro-to-DGE](https://github.com/hbctraining/Intro-to-DGE) (lesson "Wald test results" and design-formula material).

**License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Changes:** Applied to the GSE79275 genotype x treatment design (opt3-2 vs Col-0, Cd vs replete) with interaction terms; the code was rewritten for this dataset.
```

## Importing of Data from [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275)

The downloaded count matrix for [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) is downloaded to `data/geo/GSE79275/` by `data/fetch_data.sh geo`. This file was retrived from the GSE, and will be used for this experiment. As we only care about the 12 samples, we will filter for roots. We will also average all technical replicates into a single biological replicate. These steps occur below, please follow allong using the commented code. 

*Think about what this would look like if the analysis was completed within RStudio rather than Jupyter Notebook: We could see our tables more clearly, gain suggestions for syntax from the IDE, and also make sure that code is ran line by line rather than within cells.*

In [ ]:
### Bioconductor and CRAN libraries used
library(DESeq2)
library(tidyverse)
library(RColorBrewer)
library(pheatmap)
library(tximport)
library(ggplot2)
library(ggrepel)

In [ ]:
##import the counts table
DATA_DIR <- Sys.getenv("DATA_DIR", unset = "../../../data")  # repository data/ folder; run data/fetch_data.sh geo first
counts <- read.delim(file.path(DATA_DIR, "geo", "GSE79275", "GSE79275_raw_read_counts.txt"), header = TRUE, stringsAsFactors = FALSE)

# Make rownames the gene IDs (first column)
rownames(counts) <- counts$AGI
counts <- counts[ , -1]  # remove first column after setting rownames

# Ensure all counts are integers
counts <- round(as.matrix(counts))

# Get sample names from counts table
sample_names <- colnames(counts)

# Initialize vectors
genotype <- character(length(sample_names))
tissue <- character(length(sample_names))
treatment <- character(length(sample_names))
bio_rep <- integer(length(sample_names))
tech_rep <- integer(length(sample_names))

# Parse sample names
for (i in seq_along(sample_names)) {
  s <- sample_names[i]
  
  # Genotype
  if (grepl("^C", s)) {
    genotype[i] <- "col0"
  } else if (grepl("^O", s)) {
    genotype[i] <- "opt3"
  }
  
  # Tissue
  if (grepl("R", s)) {
    tissue[i] <- "Root"
  } else if (grepl("S", s)) {
    tissue[i] <- "Shoot"
  }
  
  # Treatment
  if (grepl("T", s)) {
    treatment[i] <- "Treated"
  } else {
    treatment[i] <- "Untreated"
  }
  
  # Biological replicate
  bio_rep[i] <- as.integer(sub(".*?([0-9]+)[-_][0-9]+\\.bam$", "\\1", s))
}

# Assign technical replicates sequentially for each bio_rep
colData <- data.frame(
  sample = sample_names,
  genotype = factor(genotype),
  tissue = factor(tissue),
  treatment = factor(treatment),
  bio_rep = bio_rep,
  stringsAsFactors = FALSE
)

# Order by bio_rep and assign tech_rep
colData <- colData[order(colData$bio_rep, colData$sample), ]
colData$tech_rep <- ave(colData$bio_rep, colData$bio_rep, FUN = seq_along)

# Set row names to sample names for DESeq2
rownames(colData) <- colData$sample
colData$sample <- NULL

# Inspect
colData

# Move to appropriate naming
data <- counts
meta <- colData

In [ ]:
#check inputs
data
meta

In [ ]:
# Reorder metadata rows to match the count matrix
meta <- meta[colnames(data), ]
all(colnames(data) %in% rownames(meta))
all(colnames(data) == rownames(meta))

In [ ]:
#Combine technical replicates

# Add a column identifying unique biological samples
meta$bio_id <- paste(meta$genotype, meta$tissue, meta$treatment, meta$bio_rep, sep="_")

# Sum counts for technical replicates
collapsed_counts <- sapply(unique(meta$bio_id), function(id) {
  rowSums(data[, meta$bio_id == id])
})

# Make it a proper matrix
collapsed_counts <- as.matrix(collapsed_counts)

# Update metadata to include only biological replicates
collapsed_meta <- meta %>%
  distinct(bio_id, .keep_all = TRUE) %>%
  dplyr::select(-tech_rep)
rownames(collapsed_meta) <- collapsed_meta$bio_id

In [ ]:
#Now we subset for only roots: 
# Subset metadata for root samples only
root_meta <- collapsed_meta[collapsed_meta$tissue == "Root", ]

# Subset counts matrix to match only the root samples
root_counts <- collapsed_counts[, rownames(root_meta)]

# Optional: check dimensions
dim(root_counts)
dim(root_meta)

In [ ]:
root_meta

## How DESeq2 handles interactions

### Study Design
In DESeq2, the study design defines how your experimental conditions are modeled statistically. It tells DESeq2 which variables explain variation in gene expression and how to test for differential expression.

This is expressed as a design formula:
`design = ~ factor1 + factor2 + ... + factorN`
Each factor represents a biological or technical variable. DESeq2 uses this formula to build a generalized linear model (GLM) for each gene, allowing us to test how gene expression changes with respect to the variables included in the study design.

Within our RNA-Seq experiment, we want to understand the interactions between both the treatment condition and the genotype: 
- **Col-0 vs OPT3 mutation** is our genotype factor
- **Replete vs 72hr Cd Treatment** is our treatment factor

Because the sequencing had occurred at different times, we should also be concerned about batch effects. We will look out for this in the PCA plots as well. 

# Watch this video on Study Design within DESeq2: [DESeq2 Design Explained: Understanding Interaction Terms in RNA-Seq Analysis](https://youtu.be/OPt51qc5YJ8?si=nPhUysCJ0TbjoEZ9)

### Reference levels
We also need to set our reference levels carefully, as when we are running our model with DESeq2, references are automatically ordered alphabetically. If we do not set these reference levels, then our comparisons could be made incorrectly.

In [ ]:
#Now we reference our level factors: 
# Ensure Col-0 is the reference for genotype
root_meta$genotype <- relevel(root_meta$genotype, ref = "col0")

# Ensure Untreated is the reference for treatment
root_meta$treatment <- relevel(root_meta$treatment, ref = "Untreated")

## Creating a DESeq2 Object: 
In this step, we will create a DESeqDataSet object, which is the central data container used by the DESeq2 package. To build this object, we will use our raw count matrix and metadata table as inputs. The count matrix contains the gene expression values (with genes as rows and samples as columns), while the metadata table describes each sample’s experimental condition (such as genotype and treatment).

We must also specify a design formula, which defines the experimental factors we want to test for differential expression. For our dataset, we are interested in modeling gene expression changes with respect to genotype and treatment and their interaction(`genotype:treatment), so our design formula will be:

```r
design = ~ genotype + treatment + genotype:treatment
```
Since our data are already in a gene-level count matrix, we can create the `DESeq2` object directly using the `DESeqDataSetFromMatrix()` function, rather than importing counts through `tximport`. This function will use the count and metadata information to initialize a `DESeqDataSet`, or `dds`, ready for downstream normalization and differential expression analysis.

In [ ]:
dds <- DESeqDataSetFromMatrix(countData = root_counts,
                              colData = root_meta,
                              design = ~ genotype + treatment + genotype:treatment)

In our case, we have a 2x2 factorial design: genotype × treatment.
- Two genotypes: col0 and opt3
- Two treatments: Untreated (replete) and Treated(72 hr Cd treatment)
- Three biological replicates per condition

In our design `design = ~ genotype + treatment + genotype:treatment`: 
- `genotype` = main effect of genotype (opt3 vs col0)
    - How gene expression changes on average across both genotypes when treated
- `treatment` =  main effect of treatment (Treated vs Untreated)
    - How gene expression differs on average across both treatments
- `genotype:treatment` = interaction term that tests whether the effect of treatment depends on genotype
    - Identifies genes where the treatment effect is different between col0 and opt3; E.g., a gene might increase under treatment in col0 but decrease in opt3

# QC checks on the DESeq2 data
Before performing the differential expression analysis, it is a good idea to know what sources of variation are present in your data, either by exploration during the QC and/or prior knowledge. Once you know the major sources of variation, you can remove them prior to analysis or control for them in the statistical model by including them in your design formula.

This step is critical as each additional factor in your design formula reduces your power. HOWEVER, failing to include important sources of variation can give you inaccurate results. Let's normalize the count data to make fair gene comparisons between samples for our assessment. We'll make the PCA Plots to check if there are any issues with bias due to batches within our experiment that we may have to consider in our DESeq2 study design!

*NOTE: DESeq2 doesn’t actually use normalized counts; rather, it uses the raw counts and models the normalization inside the Generalized Linear Model (GLM). These normalized counts will be useful for downstream visualization of results, but cannot be used as input to DESeq2 or any other tools that perform differential expression analysis that use the negative binomial model.*

In [ ]:
#estimate size factors
dds_norm <- estimateSizeFactors(dds)

#normalize counts
normalized_counts <- counts(dds_norm, normalized=TRUE)

### Transform counts for data visualization
vsd <- vst(dds, blind=TRUE)

In [ ]:
#check column names to see treatment groups
colnames(colData(dds))

In [ ]:
### Plot PCA 
plotPCA(vsd, intgroup="genotype")

In [ ]:
plotPCA(vsd, intgroup="treatment")

We saw this in Moudle 4, but the samples cluster well. We will not consider batch effects for the sake of simplicity. 

## Running DESeq2

A design formula tells the statistical software which sources of variation to test for. This includes both your factor of interest as well as any additional covariates that are sources of variation. For example, if you know that genotype is a significant source of variation in your data, then genotype should be included in your model. The design formula should have all of the factors in your metadata that account for major sources of variation in your data.

**In our data, it seems that 90% of our variance is captured across the genotype and treatment, and the batch effects seem to be minimal, so we can assume that we will use `design = ~ genotype + treatment + genotype:treatment`.** We created this in our DESeq2 object.

In [ ]:
#Filter low count genes
dds <- dds[rowSums(counts(dds)) >= 10, ]

In [ ]:
#Run DESeq on the dds object
dds <- DESeq(dds)

In [ ]:
#Check the dispersion estimates
plotDispEsts(dds)

##  Exploring Results (Wald test):

By default, DESeq2 uses the Wald test to identify genes that are differentially expressed between two sample classes. Given the factor(s) used in the design formula and how many factor levels are present, we can extract results for a number of different comparisons. Here, we will walk through how to obtain results from the dds object and provide some explanations on how to interpret them.

We are using a full factorial design, so `design = ~ genotype + treatment + genotype:treatment`

This means DESeq2 will model main effects of genotype and treatment, as well as all interactions between them (genotype:treatment).

We can check all of these results with: `resultsNames(dds)`

In [ ]:
resultsNames(dds)

Let's gather the effects of each treatment: 

In [ ]:
res_treatment <- results(dds, name="treatment_Treated_vs_Untreated")
res_genotype <- results(dds, name="genotype_opt3_vs_col0")
res_interaction <- results(dds, name="genotypeopt3.treatmentTreated")

For this interpretation, we are mainly interested in the effects of our treatment, exposure to cadmium for 72hrs, on the roots of the opt3 and col-0 plants. We want to see the interaction of this cadmium treatment across our opt3 mutants and across our col-0 controls. 

In our dataset, we are primarily interested in the effects of cadmium treatment (72 hrs) on the roots of Col-0 controls and opt3-2 mutants. To do this, we want to see both:

The treatment effect within Col-0 roots (Treated vs. Untreated).

The treatment effect within opt3-2 roots (Treated vs. Untreated).

Our results have these interactions:
| Coefficient                     | Meaning                | Comparison                                                                  |
|---------------------------------|------------------------|----------------------------------------------------------------------------|
| Intercept                        | Baseline expression    | Col-0 Untreated                                                            |
| genotype_opt3_vs_col0            | Main effect of genotype| opt3 vs Col-0 (Untreated)                                                 |
| treatment_Treated_vs_Untreated   | Main effect of treatment| Treated vs Untreated (Col-0)                                              |
| genotypeopt3.treatmentTreated    | Interaction effect     | (opt3 Treated - opt3 Untreated) - (Col-0 Treated - Col-0 Untreated)      |


Optionally, the interaction effect between genotype and treatment is used to determine if the cadmium response differs between opt3-2 and Col-0 roots.

When we initially created our `dds` object, we provided the design formula:
`design = ~ genotype + treatment + genotype:treatment`

This design captures the main effects of genotype and treatment, as well as all their interaction. To extract the comparisons of interest, we need to specify contrasts when using the results() function.

*NOTE: If we run the results() function without specifying contrast or name, it will return the comparison of the last level of the last variable in the design formula over the first level of this variable. If the order of levels is not specified, they are ordered alphabetically by DESeq2.*

Contrasts can be specified in two different ways (with the first method more commonly used):

- Contrasts can be supplied as a character vector with exactly three elements: the name of the factor (of interest) in the design formula, the name of the two factor levels to compare. The factor level given last is the base level for the comparison. The syntax is given below:
```
contrast <- c("factor_name", "level_of_interest", "base_level")
results(dds, contrast = contrast)
```

- Contrasts can be given as a list of 2 character vectors: the names of the fold changes for the level of interest, and the names of the fold changes for the base level. These names should match identically to the elements of resultsNames(object). This method can be useful for combining interaction terms and main effects.
```
resultsNames(dds)  # check available names
contrast_interaction <- list("treatmentTreated_vs_Untreated", "genotypeopt3-2.tissueRoot.treatmentTreated")
results(dds, contrast = contrast_interaction)
```
Alternatively, if you only had two factor levels you could do nothing and not worry about specifying contrasts (i.e. results(dds)). In this case, DESeq2 will choose what your base factor level based on alphabetical order of the levels.



    Does it matter what I choose to be my base level?

Yes, it does matter. Deciding what level is the base level will determine how to interpret the fold change that is reported. So for example, if we observe a log2 fold change of -2 this would mean the gene expression is lower in factor level of interest relative to the base level. Thus, if leaving it up to DESeq2 to decide on the contrasts be sure to check that the alphabetical order coincides with the fold change direction you are anticipating.


In [ ]:
## Save the unshrunken results to compare
res_treatment_unshrunken <- res_treatment 
res_genotype_unshrunken <- res_genotype 
res_interaction_unshrunken <- res_interaction 

# Apply fold change shrinkage
res_genotype  <- lfcShrink(dds, coef="genotype_opt3_vs_col0", type="apeglm")
res_treatment <- lfcShrink(dds, coef="treatment_Treated_vs_Untreated", type="apeglm")
res_interaction <- lfcShrink(dds, coef="genotypeopt3.treatmentTreated", type="apeglm")

## Summarizing and plotting results to compare across treatments

We will now examine the results from these experiments using a series of plots. The summaries included show how many genes change significantly according to the GLM model for the factors. The Volcano plots and heat maps show the top 30 DEGs sorted by p-value. To explore the relationships between factors, you should create a data frame that compares the differentially expressed genes (DEGs) identified for each factor. This will allow you to assess overlaps and evaluate how genotype, treatment, and their interaction influence gene expression in each comparison. Write your response below: 

To summarize the results table, a handy function in `DESeq2` is `summary()`. Confusingly, it has the same name as the function used to inspect data frames. This function, when called with a DESeq results table as input, will summarize the results using a default threshold of `padj < 0.1`. However, since we had set the alpha argument to 0.05 when creating our results table,e threshold: FDR < 0.05 (padj/FDR is used even though the output says `p-value < 0.05`). 

In [ ]:
summary(res_treatment, alpha = 0.05)
summary(res_genotype, alpha = 0.05)
summary(res_interaction, alpha = 0.05)

## 8. Visualizing the results<a id="8"></a>

We now have our list of DE genes from our results data frame for the `genotypeopt3.treatmentTreated = (opt3 Treated - opt3 Untreated) - (Col-0 Treated - Col-0 Untreated)`
comparison. What this will tell us is the interaction effect between our two combined treatments: 
- (opt3 Treated - opt3 Untreated): The effect of cadmium treatment on the opt3 mutants.
- (Col-0 Treated - Col-0 Untreated): The effect of cadmium treatment on the Col-0 controls.
- Interaction effect: The difference between these two treatment effects.

Interpretation of interaction effect example:

- Positive log2 fold change → the gene responds more strongly to cadmium in opt3 roots than in Col-0 roots.

- Negative log2 fold change → the gene responds less strongly to cadmium in opt3 roots than in Col-0 roots.

- Near zero → the treatment effect is similar between genotypes.

When we are working with large amounts of data, it can be useful to display that information graphically to gain more insight. During this lesson, we will get you started with some basic and more advanced plots commonly used when exploring differential gene expression data; however, many of these plots can be helpful in visualizing other types of data as well.

We will be working with three different data objects that we have already created earlier :

1. Metadata for our samples (a dataframe): `root_meta`
2. Normalized expression data for every gene in each of our samples (a matrix): `normalized_counts`
3. Tibble versions of the DESeq2 results we generated: `res_tableOE_tb`

First, let’s create a metadata tibble from the data frame (don’t lose the row names!)

In [ ]:
library(dplyr)
library(tibble)

make_sig_table <- function(res, alpha = 0.05) {
    res %>%
        as.data.frame() %>%
        rownames_to_column(var = "gene") %>%
        as_tibble() %>%
        filter(!is.na(padj), padj < alpha)
}

In [ ]:
sig_treatment   <- make_sig_table(res_treatment,   alpha = 0.05)
sig_genotype    <- make_sig_table(res_genotype,    alpha = 0.05)
sig_interaction <- make_sig_table(res_interaction, alpha = 0.05)


Next, let’s bring in a column with gene symbols to the normalized_counts object, so we can use them to label our plots. AGI are great for many things, but the gene symbols are much more recognizable to us, as biologists.

In [ ]:
library(biomaRt)
library(dplyr)
library(tibble)

annotate_genes_biomart <- function(sig_table) {
    
    # Extract gene IDs
    deg_genes <- sig_table$gene
    
    # Connect to Ensembl Plants (Arabidopsis)
    mart <- useMart(
        biomart = "plants_mart",
        dataset = "athaliana_eg_gene",
        host = "https://plants.ensembl.org"
    )
    
    # Retrieve gene info
    gene_info <- getBM(
        attributes = c("ensembl_gene_id", "external_gene_name"),
        filters = "ensembl_gene_id",
        values = deg_genes,
        mart = mart
    )
    
    # Mapping vector: ensembl → gene symbol
    gene_map <- setNames(gene_info$external_gene_name, gene_info$ensembl_gene_id)
    
    # Add annotation column
    sig_annotated <- sig_table %>%
        mutate(common_name = gene_map[gene]) %>%
        mutate(common_name = ifelse(is.na(common_name), gene, common_name))
    
     return(list(
        annotated_table = sig_annotated,
        gene_map = gene_map
      ))
}


In [ ]:
# Annotate
treatment_annot   <- annotate_genes_biomart(sig_treatment)
genotype_annot    <- annotate_genes_biomart(sig_genotype)
interaction_annot <- annotate_genes_biomart(sig_interaction)

In [ ]:
# Order DEGs by padj and select top 30
top30_genes <- head(rownames(res_interaction[order(res_interaction$padj), ]), 30)

# Loop through top 30 genes
for (gene_id in top30_genes) {
  
  print(plotCounts(dds,
                   gene = gene_id,
                   intgroup = c("genotype", "treatment"),
                   returnData = FALSE))  # set TRUE if you want to extract data instead
}


## Volcano plot

The above plot would be great to look at the expression levels of a good number of genes, but for more of a global view, there are other plots we can draw. A commonly used one is a volcano plot, in which you have the log-transformed adjusted p-values plotted on the y-axis and log2 fold change values on the x-axis.

To generate a volcano plot, we first need to have a column in our results data indicating whether or not the gene is considered differentially expressed based on p-adjusted values, and we will include a log2fold change here.

In [ ]:
library(ggplot2)
library(ggrepel)
library(dplyr)

plot_volcano_with_labels <- function(annot_list, title = "Volcano Plot") {
  
  res_table <- annot_list$annotated_table
  gene_map  <- annot_list$gene_map
  
  # Ensure symbol column exists
  if (!"symbol" %in% colnames(res_table)) {
    res_table <- res_table %>%
      mutate(symbol = gene_map[gene])
  }
  
  # Add empty label column
  res_table <- res_table %>% mutate(genelabels = "")
  
  # Sort by significance
  res_table <- res_table %>% arrange(padj)
  
  # Label the top 10 genes
  n_top <- min(10, sum(!is.na(res_table$padj)))
  res_table$genelabels[1:n_top] <- as.character(res_table$symbol[1:n_top])
  
  # Threshold if missing
  if (!"threshold" %in% colnames(res_table)) {
    res_table <- res_table %>%
      mutate(threshold = padj < 0.05 & abs(log2FoldChange) > 1)
  }
  
  # Plot
  ggplot(res_table, aes(x = log2FoldChange, y = -log10(padj))) +
    geom_point(aes(color = threshold)) +
    geom_text_repel(aes(label = genelabels)) +
    ggtitle(title) +
    xlab("log2 fold change") +
    ylab("-log10 adjusted p-value") +
    theme(
      legend.position = "none",
      plot.title = element_text(size = rel(1.5), hjust = 0.5),
      axis.title  = element_text(size = rel(1.25))
    )
}

In [ ]:
# Volcano plots
plot_volcano_with_labels(treatment_annot,   "Treatment Volcano")
plot_volcano_with_labels(genotype_annot,    "Genotype Volcano")
plot_volcano_with_labels(interaction_annot, "Interaction Volcano")

### Heatmap

In addition to plotting subsets, we could also extract the normalized values of all the significant genes and plot a heatmap of their expression using `pheatmap()`.

In [ ]:
library(pheatmap)
topgenes <- head(order(res_genotype$padj), 30)
pheatmap(assay(vst(dds))[topgenes, ])

In [ ]:
library(pheatmap)
topgenes <- head(order(res_treatment$padj), 30)
pheatmap(assay(vst(dds))[topgenes, ])

In [ ]:
library(pheatmap)
topgenes <- head(order(res_interaction$padj), 30)
pheatmap(assay(vst(dds))[topgenes, ])

### Treatment response curves
When we have a factorial experiment with genotype and treatment, we are often interested in how gene expression responds to a treatment in each genotype. A treatment response curve is a plot showing the change in expression of a gene (or set of genes) across treatment conditions, typically:
- Y-axis: normalized gene expression (e.g., VST counts, log2 counts)
- X-axis: treatment condition (e.g., control vs infected)

If the effect of treatment is different between genotypes, you will see non-parallel lines on the plot. Diverging or crossing lines indicate a genotype-specific response, i.e., an interaction effect.

In [ ]:
#VST Transform data
vsd <- vst(dds, blind=TRUE)

In [ ]:
plot_gene_response <- function(dds, vsd, gene) {
    library(ggplot2)
    library(dplyr)

    # Build data frame
    df <- as.data.frame(colData(dds))
    df$sample <- rownames(df)
    df$counts <- assay(vsd)[gene, ]

    # Plot genotype-specific treatment curves
    p <- ggplot(df, aes(x = treatment,
                        y = counts,
                        color = genotype,
                        group = genotype)) +
        geom_point(size = 3) +
        geom_line(aes(group = genotype), linewidth = 1) +
        theme_bw() +
        labs(
            title = paste("Treatment-response for:", gene),
            y = "VST-normalized expression"
        ) +
        theme(
            plot.title = element_text(hjust = 0.5, size = 14),
            axis.text = element_text(size = 12),
            axis.title = element_text(size = 13)
        )

    return(p)
}

In [ ]:
top5 <- head(rownames(res_interaction)[order(res_interaction$padj)], 5)

for (g in top5) {
    print(plot_gene_response(dds, vsd, g))
}